# 消息 DAG：按样本和目标查看

图不使用标签构建。红=幻觉，绿=正常，灰=特殊。可切换来源和物理 head；显示覆盖率说明稀疏边保留了多少贡献。

In [ ]:
from pathlib import Path
import json, sys
PROJECT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p/'experiments/reanchor_flow/message_dag').is_dir())
sys.path.insert(0, str(PROJECT))
OUTPUT = PROJECT/'experiments/reanchor_flow/outputs/attention_audit_v3/message_dag'
index = json.loads((OUTPUT/'index.json').read_text())
[(e['split'], e['task_type'], e['sample_id'], e['targets']) for e in index['samples']]

In [ ]:
SPLIT, TASK, SAMPLE_ID = 'test', 'QA', None
rows = [e for e in index['samples'] if e['split']==SPLIT and e['task_type']==TASK]
e = next(e for e in rows if SAMPLE_ID is None or str(e['sample_id'])==str(SAMPLE_ID))
folder = OUTPUT/e['folder']
ready = [t for t in e['targets'] if (folder/f'target_{t}.npz').exists()]
print('sample:', e['sample_id'], 'completed targets:', ready)

In [ ]:
TARGET = ready[0]  # 改为上方已完成的任意目标位置
from experiments.reanchor_flow.message_dag.view import render_target
from IPython.display import IFrame, display
import base64
page = render_target(folder, TARGET)
url = 'data:text/html;base64,' + base64.b64encode(page.read_bytes()).decode()
display(IFrame(url, width='100%', height=1100))

In [ ]:
# 整体逐 layer/head 的 N/H 统计，而不是只看一个样本
from IPython.display import Image
figure = OUTPUT/'cohorts'/f'{SPLIT}_{TASK}_heads.png'
display(Image(filename=str(figure)))